# 03 · Calidad de datos profunda y EDA
 miramos **cada una de las 20 series** (producto × tienda) por separado: cuántos ceros tiene, si hay valores raros, cómo se mueve el precio y qué patrones (semana, año, eventos) se repiten.

**Reglas del notebook**
- Se explora **solo `train`**. `valid` se abre una única vez, al final, para comprobar si se parece a `train` (sin tomar decisiones con ella). `reserva` no se toca (solo se prueba que la conversión de tipos funcione).
- Todo umbral que se calcula con datos sale **solo de `train`**.
- Las observaciones se escribieron **después de ver los números**. Lo que no se verificó, se dice.
- Las decisiones con alternativa descartada están en `decisions.md` (resumen) y `decisions_detalle.md` (completo).

## 0. Preparación
Rutas con `Path` y las columnas importantes en constantes, así las funciones de abajo no tienen valores fijos adentro y se pueden pasar a `04_scripts/`.

Una **serie** es una combinación de tienda y producto (`store_id` + `item_id`). No usamos solo `item_id` porque el mismo producto se vende en varias tiendas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 160)

DATOS = (Path("..") / "02_datos").resolve()
RUTA_TRAIN = DATOS / "03_Entrenamiento" / "train.parquet"
RUTA_VALID = DATOS / "02_Validacion" / "validacion.parquet"
RUTA_RESERVA = DATOS / "02_Validacion" / "reserva.parquet"

CLAVE_SERIE = ["store_id", "item_id"]   # identifica una serie (no asume 1 estado/depto/categoría)
COL_FECHA = "date"
COL_VENTAS = "ventas"
COL_PRECIO = "sell_price"


def cargar_tablon(ruta):
    """Lee un parquet (trae `date` como índice) y devuelve `date` como columna."""
    return pd.read_parquet(ruta).reset_index()


def etiqueta_serie(df, clave=CLAVE_SERIE):
    """Texto corto por fila para rotular series, ej. 'FOODS_3_090 @ CA_3'."""
    return df[clave[1]].astype(str) + " @ " + df[clave[0]].astype(str)


train_raw = cargar_tablon(RUTA_TRAIN)
print(train_raw.shape)

### 0.1 Tipos: `year`, `month` y `wday` pasan a categóricas
`year`, `month` y `wday` son números, pero **no se pueden sumar ni promediar con sentido** (mes 12 no es "el doble" de mes 6). Son categorías.

¿Qué diferencia hay entre `str`, `object` y `category`? (en este entorno hay **pandas 3**)

| Tipo | Qué es | Cuándo conviene |
|---|---|---|
| `object` | Contenedor genérico de objetos Python. Antes de pandas 3 era donde vivía el texto. Lento y ocupa mucha memoria. | Casi nunca para texto. |
| `str` | El tipo **de texto** de pandas 3 (reemplaza al `object` para strings). Solo guarda texto, valida y es más eficiente. | Texto libre: nombres, descripciones. |
| `category` | Guarda una **tabla de categorías** y por cada fila un número chico que apunta a ella. Las categorías son un conjunto cerrado y pueden tener **orden**. | Pocas etiquetas que se repiten (mes, día, tienda). |

**Uso `category` ordenada**, con etiquetas de texto: `month` = `"01".."12"` (así ordena bien), `wday` = `"1".."7"` y `year` = `"2013".."2015"`. Queda sin one-hot (eso es feature engineering, viene después).

In [ ]:
def a_categoricas(df, categorias, formatos=None):
    """Convierte columnas numéricas que en realidad son categorías a `category` ordenada.

    categorias: dict {columna: lista ordenada de etiquetas de texto}.
    Si aparece un valor que no está declarado, falla en vez de dejar un NaN silencioso
    (importante cuando lleguen datos nuevos, por ejemplo un año 2016).
    """
    formatos = formatos or {}
    salida = df.copy()
    for col, cats in categorias.items():
        fmt = formatos.get(col, "{}")
        etiquetas = salida[col].map(fmt.format)
        desconocidas = set(etiquetas) - set(cats)
        if desconocidas:
            raise ValueError(f"'{col}' tiene valores no declarados: {sorted(desconocidas)}")
        salida[col] = pd.Categorical(etiquetas, categories=cats, ordered=True)
    return salida


FORMATOS = {"year": "{:d}", "month": "{:02d}", "wday": "{:d}"}
CATEGORIAS = {
    "month": [f"{m:02d}" for m in range(1, 13)],
    "wday": [str(d) for d in range(1, 8)],
    # el vocabulario de años se aprende SOLO de train
    "year": sorted(train_raw["year"].map(FORMATOS["year"].format).unique()),
}

train = a_categoricas(train_raw, CATEGORIAS, FORMATOS)

# Misma conversión en valid y reserva: acá solo miramos los tipos, no los datos
tipos = {"train": train}
for nombre, ruta in {"valid": RUTA_VALID, "reserva": RUTA_RESERVA}.items():
    tipos[nombre] = a_categoricas(cargar_tablon(ruta), CATEGORIAS, FORMATOS)
pd.DataFrame({n: d[["year", "month", "wday"]].dtypes.astype(str) for n, d in tipos.items()})

In [ ]:
# Cómo se ven las categorías y cuánta memoria ocupa el mismo texto en cada tipo
print(train["month"].dtype)
print(train["month"].cat.categories.tolist())

mem = pd.Series({
    "object": train["weekday"].astype(object).memory_usage(deep=True),
    "str": train["weekday"].astype("str").memory_usage(deep=True),
    "category": train["weekday"].astype("category").memory_usage(deep=True),
}, name="bytes")
print(mem)

# wday sigue la convención de M5: 1 = sábado. Lo verificamos contra `weekday`
pd.crosstab(train["wday"], train["weekday"], margins=False)

**Observación:**
- `year`, `month` y `wday` ya son `category` ordenada en `train`, `valid` y `reserva` (de `reserva` solo se miraron los tipos).
- Para la columna `weekday` (20.809 filas): `object` ocupa 1.168.403 bytes, `str` 315.234 y `category` 21.048. Con `category` pesa unas 55 veces menos que con `object`.
- `wday` sigue la convención de M5: **1 = sábado, 2 = domingo, 3 = lunes… 7 = viernes**. El cruce con `weekday` confirma que cada número corresponde a un único día. Por eso el orden `"1".."7"` arranca en sábado, no en lunes.
- `year` se aprende de `train` (2013-2015). Si llega un 2016, la función **falla** a propósito en vez de dejar un nulo en silencio.

## 1. Calidad de datos por serie
Ya sabemos que no hay nulos ni duplicados (notebook 01/02). Ahora vamos **serie por serie**.

### 1.1 Cobertura
¿Cada serie tiene un dato por día, sin huecos? Un hueco rompe los rezagos y las medias móviles.

In [ ]:
def cobertura_por_serie(df, clave=CLAVE_SERIE, col_fecha=COL_FECHA):
    """Filas, rango de fechas y huecos de cada serie contra el calendario del df."""
    inicio, fin = df[col_fecha].min(), df[col_fecha].max()
    esperados = (fin - inicio).days + 1
    r = df.groupby(clave, observed=True)[col_fecha].agg(filas="size", dias_unicos="nunique", desde="min", hasta="max")
    r["duplicadas"] = r["filas"] - r["dias_unicos"]
    r["huecos_internos"] = (r["hasta"] - r["desde"]).dt.days + 1 - r["dias_unicos"]
    r["dias_antes_de_empezar"] = (r["desde"] - inicio).dt.days
    r["dias_tras_terminar"] = (fin - r["hasta"]).dt.days
    r["cobertura_%"] = (100 * r["dias_unicos"] / esperados).round(1)
    return r, esperados


def fechas_faltantes(df, col_fecha=COL_FECHA):
    """Días del calendario que no aparecen en ninguna serie."""
    calendario = pd.date_range(df[col_fecha].min(), df[col_fecha].max(), freq="D")
    return calendario.difference(pd.DatetimeIndex(df[col_fecha].unique()))


cob, dias_esperados = cobertura_por_serie(train)
print("Días esperados en train:", dias_esperados, "| fechas que faltan en TODAS las series:", len(fechas_faltantes(train)))
cob.sort_values("cobertura_%").head(6)

In [ ]:
# Resumen: ¿alguna serie con huecos internos, duplicados o que empiece tarde?
cob[["duplicadas", "huecos_internos", "dias_antes_de_empezar", "dias_tras_terminar"]].describe().loc[["min", "max"]]

**Observación:**
- `train` tiene 1.044 días (2013-01-01 a 2015-11-10). Ninguna fecha falta en todas las series a la vez, y no hay duplicados ni huecos dentro de ninguna serie.
- 18 de las 20 series tienen los 1.044 días. Las dos de `FOODS_3_288` empiezan tarde: el 2013-02-02 en `CA_3` (cobertura 96,9 %) y el 2013-02-09 en `CA_4` (96,3 %). Son los 71 días que ya se eliminaron en el notebook 02.
- La cobertura de fechas está bien. Pero que haya una fila por día **no** significa que el producto estuviera disponible: eso se ve en los ceros (siguiente punto).

### 1.2 Ceros e intermitencia
Un cero en `ventas` puede ser "nadie compró" (demanda real) o "no había stock / no se vendía". No lo podemos separar con estos datos, así que **medimos cuánto pesan** los ceros en cada serie.

Para clasificar la demanda usamos dos números (clasificación clásica de Syntetos-Boylan):
- **ADI**: cada cuántos días hay una venta (1 = vende todos los días).
- **CV²**: qué tan variable es la cantidad cuando sí se vende (coeficiente de variación al cuadrado).

Umbrales **de la bibliografía** (1,32 y 0,49), no aprendidos de estos datos: `smooth` (regular), `erratic` (errática), `intermittent` (intermitente), `lumpy` (intermitente y errática).

In [ ]:
def racha_maxima(mascara):
    """Largo de la racha más larga de True seguidos."""
    m = np.asarray(mascara, dtype=int)
    if not m.any():
        return 0
    cambios = np.diff(np.concatenate(([0], m, [0])))
    return int((np.flatnonzero(cambios == -1) - np.flatnonzero(cambios == 1)).max())


def intermitencia_por_serie(df, clave=CLAVE_SERIE, col_fecha=COL_FECHA, col_ventas=COL_VENTAS,
                            corte_adi=1.32, corte_cv2=0.49):
    """% de ceros, racha máxima de ceros, ADI, CV² y tipo de demanda por serie."""
    filas = []
    for k, g in df.sort_values(col_fecha).groupby(clave, observed=True):
        v = g[col_ventas]
        pos = v[v > 0]
        adi = len(v) / len(pos) if len(pos) else np.inf
        cv2 = (pos.std(ddof=0) / pos.mean()) ** 2 if len(pos) else np.nan
        if adi < corte_adi:
            tipo = "smooth" if cv2 < corte_cv2 else "erratic"
        else:
            tipo = "intermittent" if cv2 < corte_cv2 else "lumpy"
        filas.append({**dict(zip(clave, k)), "dias": len(v), "pct_ceros": 100 * (v == 0).mean(),
                      "racha_max_ceros": racha_maxima(v == 0), "media": v.mean(),
                      "ADI": adi, "CV2": cv2, "tipo": tipo})
    return pd.DataFrame(filas).set_index(clave)


inter = intermitencia_por_serie(train)
inter.sort_values("pct_ceros", ascending=False).round(2)

In [ ]:
print(inter["tipo"].value_counts().to_string())
print()
# ¿Hay días en que TODA una tienda vende 0 (tienda cerrada / dato faltante)?
ventas_tienda_dia = train.groupby(["store_id", COL_FECHA], observed=True)[COL_VENTAS].sum()
cerradas = ventas_tienda_dia[ventas_tienda_dia == 0]
print("Días con toda la tienda en 0:", len(cerradas))
cerradas.reset_index()[["store_id", COL_FECHA]].assign(dia=lambda d: d[COL_FECHA].dt.day_name()).head(10)

In [ ]:
def rachas_de_ceros(df, minimo, clave=CLAVE_SERIE, col_fecha=COL_FECHA, col_ventas=COL_VENTAS, col_precio=COL_PRECIO):
    """Rachas de ceros seguidos de al menos `minimo` días: inicio, fin, largo y precio vigente."""
    filas = []
    for k, g in df.sort_values(col_fecha).groupby(clave, observed=True):
        z = (g[col_ventas] == 0).to_numpy().astype(int)
        cambios = np.diff(np.concatenate(([0], z, [0])))
        for a, b in zip(np.flatnonzero(cambios == 1), np.flatnonzero(cambios == -1)):
            if b - a >= minimo:
                filas.append({**dict(zip(clave, k)), "desde": g[col_fecha].iloc[a].date(), "hasta": g[col_fecha].iloc[b - 1].date(),
                              "dias": b - a, "precio_en_la_racha": g[col_precio].iloc[a:b].iloc[0]})
    return pd.DataFrame(filas)


# Rachas de 25 días o más: cuatro semanas sin vender nada ya es raro para "nadie compró"
rachas = rachas_de_ceros(train, minimo=25)
print("Rachas de 25+ días:", len(rachas), "| de 100+ días:", int((rachas["dias"] >= 100).sum()))
rachas.sort_values("dias", ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
orden = inter.sort_values("pct_ceros").reset_index()
orden["serie"] = etiqueta_serie(orden)
axes[0].barh(orden["serie"], orden["pct_ceros"])
axes[0].set_title("% de días con ventas = 0 por serie")
axes[0].set_xlabel("% de días")

sns.scatterplot(data=inter.reset_index(), x="ADI", y="CV2", hue="tipo", s=70, ax=axes[1])
axes[1].axvline(1.32, ls="--", c="gray"); axes[1].axhline(0.49, ls="--", c="gray")
axes[1].set_title("Tipo de demanda (ADI vs CV²)")
plt.tight_layout(); plt.show()

**Observación:**
- Por tipo de demanda: 15 series son `smooth`, 4 `intermittent` y 1 `erratic`. Las 4 intermitentes tienen entre 32 % y 45 % de días en cero: `CA_4` `FOODS_3_120` (45,3 %), `CA_4` `FOODS_3_587` (41,9 %), `CA_4` `FOODS_3_329` (39,6 %) y `CA_3` `FOODS_3_329` (31,7 %). Otras 8 series tienen entre 7 % y 22 % de ceros y 8 series menos de 2,5 %.
- **Los ceros vienen en bloques, no sueltos.** Hay 5 rachas de 100 días o más: `CA_4` `FOODS_3_587` (361 días, de 2013-01-31 a 2014-01-26), `FOODS_3_329` en las dos tiendas (271-272 días, del 2015-01-30 a fines de octubre 2015, mismas fechas en `CA_3` y `CA_4`), `CA_3` `FOODS_3_120` (149 días desde el 2013-01-01) y `CA_4` `FOODS_3_120` (140 días en 2014). En todas el precio sigue cargado.
- En `CA_4`, `FOODS_3_090` (5 rachas) y `FOODS_3_202` (6 rachas) tienen rachas de 28 a 34 días, repartidas más o menos cada 3 meses. Ese ritmo regular no parece azar.
- Los únicos 4 días con **toda una tienda en cero** son el 25 de diciembre de 2013 y de 2014, en las dos tiendas (Navidad).
- **No verificado:** por qué hay esas rachas (falta de stock, producto retirado, dato faltante). Los datos no lo dicen. Una racha de 361 días no es "nadie compró": no conviene tratarla como demanda real igual que un cero suelto.

### 1.3 Valores extremos en `ventas`
Un valor "extremo" es uno muy lejos del resto **de su propia serie** (un producto que vende 3 por día no se compara con uno que vende 80). Usamos la regla de Tukey: es extremo si supera `Q3 + k·IQR` (Q3 = percentil 75, IQR = Q3 − Q1). Los límites se calculan **solo con `train`**.

Solo **marcamos**: no se borra ni se recorta nada en este notebook. Un pico puede ser un error o una venta real (fiesta, promo).

In [ ]:
def limites_extremos(df, clave=CLAVE_SERIE, col_ventas=COL_VENTAS, k=3.0):
    """Límite superior de Tukey por serie. Se calcula con el df recibido (train)."""
    q = df.groupby(clave, observed=True)[col_ventas].quantile([0.25, 0.75]).unstack()
    q.columns = ["q1", "q3"]
    q["iqr"] = q["q3"] - q["q1"]
    q["limite_sup"] = q["q3"] + k * q["iqr"]
    return q


def marcar_extremos(df, limites, clave=CLAVE_SERIE, col_ventas=COL_VENTAS):
    """Agrega `limite_sup` y `es_extremo` a cada fila usando límites ya calculados."""
    salida = df.join(limites[["limite_sup"]], on=clave)
    salida["es_extremo"] = salida[col_ventas] > salida["limite_sup"]
    return salida


lim = limites_extremos(train, k=3.0)
train_ext = marcar_extremos(train, lim)

resumen_ext = train_ext.groupby(CLAVE_SERIE, observed=True).agg(
    extremos=("es_extremo", "sum"), maximo=(COL_VENTAS, "max"), mediana=(COL_VENTAS, "median"),
    p99=(COL_VENTAS, lambda s: s.quantile(0.99)))
resumen_ext = resumen_ext.join(lim[["q3", "iqr", "limite_sup"]])
resumen_ext["max_sobre_limite"] = (resumen_ext["maximo"] / resumen_ext["limite_sup"]).round(2)
print("Mínimo de ventas:", train[COL_VENTAS].min(), "| ¿todas enteras?", bool((train[COL_VENTAS] % 1 == 0).all()))
print("Total de filas extremas:", int(train_ext["es_extremo"].sum()), "de", len(train_ext))
resumen_ext.sort_values("extremos", ascending=False).round(1)

In [ ]:
# Los 12 valores más lejanos del límite: ¿qué día es y había evento?
cols = [COL_FECHA, "store_id", "item_id", COL_VENTAS, "limite_sup", "weekday", "event_name_1"]
top = train_ext[train_ext["es_extremo"]].assign(veces_limite=lambda d: (d[COL_VENTAS] / d["limite_sup"]).round(2))
top.sort_values("veces_limite", ascending=False)[cols + ["veces_limite"]].head(12)

In [ ]:
# ¿En qué meses y con qué eventos caen los extremos?
ext = train_ext[train_ext["es_extremo"]]
print(ext.groupby(ext[COL_FECHA].dt.to_period("M")).size().sort_values(ascending=False).head(6).to_string())
print()
print("Extremos en día con evento:", int(ext["hay_evento"].sum()), "de", len(ext),
      "| % de días con evento en train:", round(100 * train["hay_evento"].mean(), 1))

**Observación:**
- Con la regla Q3 + 3·IQR por serie hay **37 valores extremos de 20.809 filas (0,18 %)**. No hay ventas negativas ni decimales (mínimo 0).
- Están muy concentrados: 15 son de `CA_3` `FOODS_3_090` y 19 de los 37 caen en agosto y septiembre de 2013. El más alto (763 el 2013-09-14) supera el límite solo 1,7 veces: son valores altos, no errores de tipeo.
- Solo 3 de los 37 caen en día con evento (8 %), igual que el 7,9 % de días con evento en `train`. Los extremos no se explican por eventos.
- En la sección 2.5 se ve de dónde vienen los de agosto-septiembre 2013: coinciden con una baja de precio. **No se borra ni recorta nada.**
- Limitación: en las series intermitentes, el IQR de una serie con muchos ceros es poco informativo.

### 1.4 Precio
En esta base el precio es **semanal** (cambia por `wm_yr_wk`). Revisamos tres cosas: si es verdad que no cambia dentro de la semana, cuánto y cuántas veces cambia cada serie, y si las semanas con precio más bajo venden más.

In [ ]:
# ¿El precio es constante dentro de cada semana de cada serie?
precios_por_semana = train.groupby(CLAVE_SERIE + ["wm_yr_wk"], observed=True)[COL_PRECIO].nunique()
print("Combinaciones serie-semana con más de 1 precio:", int((precios_por_semana > 1).sum()))
print("Precios <= 0:", int((train[COL_PRECIO] <= 0).sum()), "| nulos:", int(train[COL_PRECIO].isna().sum()))


def resumen_precio(df, clave=CLAVE_SERIE, col_fecha=COL_FECHA, col_precio=COL_PRECIO):
    """Mínimo, máximo y cambios de precio semana a semana por serie."""
    sem = (df.sort_values(col_fecha).groupby(clave + ["wm_yr_wk"], observed=True)[col_precio].first().reset_index())
    filas = []
    for k, g in sem.groupby(clave, observed=True):
        var = g[col_precio].pct_change().dropna()
        cambios = var[var != 0]
        filas.append({**dict(zip(clave, k)), "semanas": len(g), "precio_min": g[col_precio].min(),
                      "precio_max": g[col_precio].max(), "max_sobre_min": g[col_precio].max() / g[col_precio].min(),
                      "semanas_con_cambio": len(cambios), "pct_semanas_con_cambio": 100 * len(cambios) / max(len(var), 1),
                      "cambio_medio_abs_%": 100 * cambios.abs().mean() if len(cambios) else 0.0,
                      "mayor_baja_%": 100 * cambios.min() if len(cambios) else 0.0,
                      "mayor_suba_%": 100 * cambios.max() if len(cambios) else 0.0})
    return pd.DataFrame(filas).set_index(clave)


precio = resumen_precio(train)
precio.round(2)

In [ ]:
def semanal_serie(df, clave=CLAVE_SERIE, col_fecha=COL_FECHA, col_ventas=COL_VENTAS, col_precio=COL_PRECIO):
    """Ventas y precio por serie y semana. Solo semanas completas (7 días)."""
    s = df.groupby(clave + ["wm_yr_wk"], observed=True).agg(
        dias=(col_fecha, "nunique"), ventas=(col_ventas, "sum"), precio=(col_precio, "first")).reset_index()
    s = s[s["dias"] == 7].copy()
    s = s.sort_values(clave + ["wm_yr_wk"])
    g = s.groupby(clave, observed=True)
    s["var_precio"] = g["precio"].pct_change()
    s["ventas_rel"] = s["ventas"] / g["ventas"].transform("mean")      # ventas de la semana / promedio de la serie
    s["precio_rel"] = s["precio"] / g["precio"].transform("median")    # precio de la semana / mediana de la serie
    return s


sem = semanal_serie(train)

# Correlación de Spearman (por rangos) entre precio y ventas semanales, dentro de cada serie
corr = sem.groupby(CLAVE_SERIE, observed=True).apply(lambda g: g["precio"].corr(g["ventas"], method="spearman"))
corr = corr.rename("spearman_precio_ventas")
print(corr.describe().round(2).to_string())

# Semanas donde el precio bajó / igual / subió vs la semana anterior
sem["movimiento"] = pd.cut(sem["var_precio"], [-np.inf, -1e-9, 1e-9, np.inf], labels=["baja", "igual", "suba"])
sem.groupby("movimiento", observed=True).agg(semanas=("ventas_rel", "size"), ventas_rel_media=("ventas_rel", "mean")).round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.scatterplot(data=sem, x="precio_rel", y="ventas_rel", alpha=0.35, ax=axes[0])
axes[0].set_title("Semanas: precio relativo vs ventas relativas (todas las series)")
axes[0].set_xlabel("precio / mediana de la serie"); axes[0].set_ylabel("ventas / media de la serie")
axes[0].set_xlim(0.5, 1.5); axes[0].set_ylim(0, 4)
corr.sort_values().rename(lambda x: x).plot.barh(ax=axes[1])
axes[1].set_title("Spearman precio-ventas por serie")
plt.tight_layout(); plt.show()

In [ ]:
# Las semanas con baja de precio venden MENOS que el promedio. Miremos una por una
bajas = sem[sem["movimiento"] == "baja"].copy()
bajas["desde"] = bajas["wm_yr_wk"].map(train.groupby("wm_yr_wk")[COL_FECHA].min().dt.date)
print("Semanas con baja:", len(bajas), "| con ventas < 15 % del promedio de su serie:", int((bajas["ventas_rel"] < 0.15).sum()),
      "| con ventas > promedio:", int((bajas["ventas_rel"] > 1).sum()))
bajas[CLAVE_SERIE + ["desde", "precio", "var_precio", "ventas", "ventas_rel"]].sort_values("desde").round(2)

In [ ]:
# Caso de la serie más vendida (FOODS_3_090 @ CA_3): ventas semanales según el precio vigente
s090 = sem[(sem["store_id"] == "CA_3") & (sem["item_id"] == "FOODS_3_090")].copy()
s090["desde"] = s090["wm_yr_wk"].map(train.groupby("wm_yr_wk")[COL_FECHA].min())
por_precio = s090.groupby("precio").agg(semanas=("ventas", "size"), ventas_mediana=("ventas", "median"),
                                        ventas_min=("ventas", "min"), ventas_max=("ventas", "max"))
print(por_precio.round(0).to_string())
# Semanas a 1,00 con ventas altas: ¿cuándo fueron?
print()
print("Semanas a precio 1,00 con más de 500 unidades:", s090[(s090["precio"] == 1.0) & (s090["ventas"] > 500)]["desde"].dt.date.tolist())

In [ ]:
# ¿Pasa en otras series? Semanas con ventas < 15 % del promedio de su serie (excluyendo las rachas largas de ceros)
def semanas_hundidas(sem, umbral=0.15):
    """Por serie: semanas con ventas muy bajas y cuántas de ellas están al precio mínimo de la serie."""
    s = sem.copy()
    s["precio_minimo"] = s["precio"] == s.groupby(CLAVE_SERIE, observed=True)["precio"].transform("min")
    s["hundida"] = s["ventas_rel"] < umbral
    r = s.groupby(CLAVE_SERIE, observed=True).agg(semanas=("hundida", "size"), hundidas=("hundida", "sum"))
    r["hundidas_a_precio_minimo"] = s[s["hundida"] & s["precio_minimo"]].groupby(CLAVE_SERIE, observed=True).size()
    r["hundidas_a_precio_minimo"] = r["hundidas_a_precio_minimo"].fillna(0)
    # En series de precio fijo todas las semanas son "precio mínimo": el dato no informa nada
    r["precio_varia"] = s.groupby(CLAVE_SERIE, observed=True)["precio"].nunique() > 1
    r.loc[~r["precio_varia"], "hundidas_a_precio_minimo"] = np.nan
    return r


hundidas = semanas_hundidas(sem)
hundidas[hundidas["hundidas"] > 0].sort_values("hundidas", ascending=False)

**Observación:**
- El precio es constante dentro de cada semana, no hay precios en cero ni nulos.
- 4 series tienen precio fijo todo el período (`FOODS_3_120` en las dos tiendas, `CA_4` `FOODS_3_288` y `CA_4` `FOODS_3_329`). Las que más cambian son `CA_3` `FOODS_3_202` (19 semanas con cambio, 12,8 %) y `CA_3` `FOODS_3_090` (18 semanas, 12,1 %), con bajas de hasta −34 % y subas de hasta +53 %. Cuatro productos (`252`, `555`, `586`, `714`) tienen un único ajuste de +6 a +7 %, igual en ambas tiendas.
- **Relación con las ventas:** en las 16 series con precio variable, la correlación de Spearman entre precio y ventas semanales tiene mediana −0,01 y va de −0,74 a +0,36. No hay una relación consistente.
- Las 25 semanas con baja de precio venden en promedio **0,39** veces lo normal de su serie (las semanas sin cambio, 1,01). En 17 de las 25, las ventas son menos del 15 % del promedio y solo 4 superan el promedio.
- Caso `CA_3` `FOODS_3_090`: a precio 1,00 hubo 30 semanas con mediana de 30 unidades, contra medianas de 793 a 1.581 a los otros precios. Solo 8 semanas a 1,00 vendieron más de 500: las 6 del 2013-08-17 al 2013-09-21 (la promoción de la sección 2.5) y dos de noviembre de 2013. Desde diciembre de 2013, toda semana a 1,00 vendió muy poco (la mayor, 392 unidades; el resto, menos de 70).
- También se hunden semanas a precio bajo en `CA_3` `FOODS_3_202` (12 de 16 semanas hundidas están a su precio mínimo). Pero otras series se hunden **sin** cambiar de precio (`FOODS_3_120`, `CA_4` `FOODS_3_329`, `CA_4` `FOODS_3_288`), así que el precio no explica todo.
- **Hipótesis no verificada:** el precio 1,00 podría ser un valor de relleno cuando el producto no está disponible, y no un descuento real. Habría que confirmarlo con quien conoce el origen de los datos. Mientras tanto, la correlación precio-ventas mezcla promociones reales con falta de disponibilidad, y **no sirve como medida de elasticidad**.

## 2. EDA
### 2.1 Tendencia por serie
Media móvil de 28 días (4 semanas exactas, así cada día de la semana pesa igual) para cada una de las 20 series. Cada gráfico tiene su propia escala.

In [ ]:
def media_diaria_por_serie(df, clave=CLAVE_SERIE, col_fecha=COL_FECHA, col_ventas=COL_VENTAS):
    """Tabla fecha x serie con las ventas diarias (para gráficos y autocorrelación)."""
    return df.pivot_table(index=col_fecha, columns=clave, values=col_ventas, aggfunc="sum", observed=True)


ancha = media_diaria_por_serie(train)
movil = ancha.rolling(28, min_periods=28).mean()

fig, axes = plt.subplots(5, 4, figsize=(16, 13), sharex=True)
for ax, col in zip(axes.ravel(), movil.columns):
    movil[col].plot(ax=ax, lw=1.3)
    ax.set_title(f"{col[1]} @ {col[0]}", fontsize=9); ax.set_xlabel("")
plt.suptitle("Media móvil 28 días de ventas diarias", y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
def tendencia_por_serie(df, anio_a, anio_b, clave=CLAVE_SERIE, col_fecha=COL_FECHA, col_ventas=COL_VENTAS):
    """Media diaria de dos años completos y el cambio % de uno al otro, por serie."""
    d = df.assign(anio=df[col_fecha].dt.year)
    m = d[d["anio"].isin([anio_a, anio_b])].pivot_table(index=clave, columns="anio", values=col_ventas, aggfunc="mean", observed=True)
    m["cambio_%"] = 100 * (m[anio_b] / m[anio_a] - 1)
    return m


tend = tendencia_por_serie(train, 2013, 2014)
print("Series que bajaron:", int((tend["cambio_%"] < 0).sum()), "| subieron:", int((tend["cambio_%"] > 0).sum()))
tend.sort_values("cambio_%").round(1)

**Observación:**
- Comparando la media diaria de 2014 contra 2013: bajaron 12 series y subieron 8.
- Las series más estables están en `CA_3`: `FOODS_3_252` (+0,1 %), `FOODS_3_714` (−2,0 %), `FOODS_3_586` (+3,5 %) y `FOODS_3_555` (+5,1 %). Esas parecen tener tendencia plana.
- Los cambios grandes coinciden con los episodios de ceros y de precio: `CA_4` `FOODS_3_587` pasa de 1,2 a 12,3 por día (+947 %) y su racha de ceros terminó el 2014-01-26; `CA_3` `FOODS_3_120` sube 155 % y tuvo 149 días en cero al inicio de 2013. `CA_3` `FOODS_3_090` baja 42 % (177 → 103): 2013 incluye la promoción de agosto-septiembre y 2014 tiene semanas hundidas a precio 1,00.
- Por eso esas "tendencias" **no parecen tendencias de demanda**, sino cambios de disponibilidad o de promoción. No verifiqué las fechas de `CA_3` `FOODS_3_587` (+410 %), pero tiene 52 semanas hundidas, un patrón parecido.

### 2.2 Estacionalidad semanal por serie
Para cada serie: venta promedio de cada día de la semana dividida por la venta promedio de la serie. 1,0 = día normal; 1,4 = 40 % más que el promedio.

In [ ]:
ORDEN_DIAS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]


def indice_semanal(df, clave=CLAVE_SERIE, col_ventas=COL_VENTAS):
    """Venta media por día de la semana / venta media de la serie."""
    por_dia = df.groupby(clave + ["weekday"], observed=True)[col_ventas].mean().unstack("weekday")[ORDEN_DIAS]
    return por_dia.div(df.groupby(clave, observed=True)[col_ventas].mean(), axis=0)


idx_sem = indice_semanal(train)
idx_sem.index = [f"{i[1]} @ {i[0]}" for i in idx_sem.index]

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(idx_sem, annot=True, fmt=".2f", cmap="RdYlBu_r", center=1, ax=ax)
ax.set_title("Índice semanal (1 = promedio de la serie)")
plt.show()
print("Índice mediano de las 20 series por día:")
print(idx_sem.median().round(2).to_string())
print("Rango (máx/mín) por serie: mediana", round((idx_sem.max(axis=1) / idx_sem.min(axis=1)).median(), 2))
print("Series donde el sábado o el domingo es el día más alto:", int(idx_sem.idxmax(axis=1).isin(["Saturday", "Sunday"]).sum()), "de", len(idx_sem))

**Observación:**
- Índice mediano de las 20 series: lunes 0,96, martes 0,89, miércoles 0,89, jueves 0,90, viernes 0,99, **sábado 1,17, domingo 1,23**.
- En 19 de las 20 series el día más alto es sábado o domingo. Dentro de una serie, el mejor día vende en mediana 1,51 veces lo que el peor.
- El patrón semanal es parejo entre series, así que es una variable fuerte y común para todas.

### 2.3 Autocorrelación y estacionalidad anual
La **autocorrelación** en el rezago *k* mide cuánto se parece la serie a sí misma *k* días antes (1 = igual, 0 = nada que ver). Un pico en el rezago 7 significa patrón semanal; en 364 (52 semanas), patrón anual. La calculamos a mano con numpy: la banda gris es ±1,96/√n, o sea lo esperable por azar en una serie sin patrón.

Ojo: si la serie tiene tendencia, la autocorrelación baja lento y se infla. Lo tenemos en cuenta al leerla.

In [ ]:
def autocorrelacion(serie, rezagos):
    """Autocorrelación simple de una serie diaria completa, para cada rezago pedido."""
    x = np.asarray(pd.Series(serie).dropna(), dtype=float)   # series que empiezan tarde traen NaN al inicio
    x = x - x.mean()
    denominador = (x ** 2).sum()
    return np.array([(x[k:] * x[:-k]).sum() / denominador for k in rezagos])


# Total diario (suma de las 20 series), con el calendario completo
total = train.groupby(COL_FECHA)[COL_VENTAS].sum()
total = total.reindex(pd.date_range(total.index.min(), total.index.max(), freq="D"))
assert total.notna().all(), "hay días sin datos"

rez_corto = np.arange(1, 57)
rez_largo = np.arange(1, 401)
banda = 1.96 / np.sqrt(len(total))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].bar(rez_corto, autocorrelacion(total, rez_corto))
axes[0].set_title("Autocorrelación del total diario (rezagos 1-56)"); axes[0].set_xlabel("rezago (días)")
axes[1].plot(rez_largo, autocorrelacion(total, rez_largo))
axes[1].set_title("Hasta 400 días"); axes[1].set_xlabel("rezago (días)")
for ax in axes:
    ax.axhspan(-banda, banda, color="gray", alpha=0.2)
for k in (7, 14, 21, 28, 35, 42, 49, 56):
    axes[0].axvline(k, color="red", alpha=0.2)
for k in (364,):
    axes[1].axvline(k, color="red", alpha=0.4)
plt.tight_layout(); plt.show()

pd.Series(autocorrelacion(total, [1, 2, 3, 7, 14, 28, 364]), index=["1", "2", "3", "7", "14", "28", "364"], name="autocorr total").round(2)

In [ ]:
# Lo mismo por serie, solo en rezagos clave
REZAGOS = [1, 7, 14, 28, 364]
ac = pd.DataFrame({f"{c[1]} @ {c[0]}": autocorrelacion(ancha[c], REZAGOS) for c in ancha.columns}, index=[f"lag {k}" for k in REZAGOS]).T

fig, ax = plt.subplots(figsize=(7, 7))
sns.heatmap(ac, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-0.6, vmax=0.6, ax=ax)
ax.set_title("Autocorrelación por serie")
plt.show()
print("Banda de azar (±):", round(banda, 3))
print("Series con autocorrelación en lag 7 sobre la banda:", int((ac['lag 7'] > banda).sum()), "de", len(ac),
      "| en lag 364:", int((ac['lag 364'] > banda).sum()))
print(ac.describe().loc[["min", "50%", "max"]].round(2).to_string())

In [ ]:
# Estacionalidad anual: venta diaria promedio del total, por mes y año (se omite nov-2015 por estar incompleto)
m = total.to_frame("ventas")
m["anio"], m["mes"] = m.index.year, m.index.month
m = m[~((m["anio"] == 2015) & (m["mes"] == 11))]
anual = m.groupby(["anio", "mes"])["ventas"].mean().unstack("anio")

fig, ax = plt.subplots(figsize=(10, 4))
anual.plot(marker="o", ax=ax)
ax.set_title("Venta diaria promedio (total) por mes"); ax.set_xlabel("mes"); ax.set_xticks(range(1, 13))
plt.show()
anual.round(1)

**Observación:**
- **Semanal:** el total diario tiene autocorrelación 0,63 en el rezago 7, 0,53 en el 14 y 0,40 en el 28; el rezago 3 solo 0,25. Las 20 series superan la banda de azar (±0,061) en el rezago 7; la mediana es 0,43 y la más baja 0,11. Los rezagos 7, 14 y 28 son candidatos naturales para variables del modelo.
- La autocorrelación baja despacio (0,69 en el rezago 1). Parte de eso es el nivel que cambia a lo largo del tiempo en algunas series (rachas de ceros, promos), no solo memoria de corto plazo.
- **Anual:** el total tiene 0,29 en el rezago 364, pero por serie la mediana es 0,11 y solo 11 de 20 superan la banda. Con 2,9 años de datos hay solo ~2 ciclos para comparar, y la señal anual es débil.
- Por mes: **julio** es alto los tres años (661, 670 y 652 por día) y **diciembre** bajo en los dos años observados (472 y 427). El resto no se repite: septiembre 2013 es 880 contra 629 en 2014; noviembre es 464 en 2013 y 643 en 2014.
- Con estos datos no puedo separar una estacionalidad anual real de los efectos de promociones y rachas de ceros.

### 2.4 Efecto de los eventos por producto
Comparar el promedio de días con evento contra días sin evento puede engañar: si un evento cae siempre en domingo, parece que "sube" solo porque es domingo. Por eso lo comparamos **contra lo esperado para ese mismo día de la semana y esa misma serie** (promedio de días sin evento).

`razón = ventas reales / ventas esperadas`. 1,00 = el evento no cambia nada; 1,20 = 20 % más.

Para saber si una razón es ruido, la comparamos con la que sale de elegir **días sin evento al azar** (mismo número de días, 1000 sorteos).

In [ ]:
def esperado_sin_evento(df, clave=CLAVE_SERIE, col_ventas=COL_VENTAS):
    """Agrega `esperado`: media de la serie en ese día de la semana, usando solo días sin evento."""
    base = df[~df["hay_evento"]].groupby(clave + ["weekday"], observed=True)[col_ventas].mean().rename("esperado")
    return df.join(base, on=clave + ["weekday"])


def razon_eventos(df, por, clave=CLAVE_SERIE, col_ventas=COL_VENTAS):
    """Razón ventas/esperado agrupando por una o más columnas, solo en días con evento."""
    d = esperado_sin_evento(df, clave, col_ventas)
    ev = d[d["hay_evento"]]
    g = ev.groupby(por, observed=True).agg(ventas=(col_ventas, "sum"), esperado=("esperado", "sum"), dias_evento=(COL_FECHA, "nunique"))
    g["razon"] = g["ventas"] / g["esperado"]
    return g[["dias_evento", "razon"]]


def intervalo_azar(df, n_dias, clave=CLAVE_SERIE, col_ventas=COL_VENTAS, sorteos=1000, semilla=0):
    """Rango 2,5 %-97,5 % de la razón al elegir `n_dias` días SIN evento al azar."""
    d = esperado_sin_evento(df, clave, col_ventas)
    d = d[~d["hay_evento"]].groupby(COL_FECHA).agg(v=(col_ventas, "sum"), e=("esperado", "sum"))
    rng = np.random.default_rng(semilla)
    idx = np.array([rng.choice(len(d), size=n_dias, replace=False) for _ in range(sorteos)])
    razones = d["v"].to_numpy()[idx].sum(axis=1) / d["e"].to_numpy()[idx].sum(axis=1)
    return np.percentile(razones, [2.5, 97.5])


# Navidad se saca de estas tablas: las tiendas están cerradas (ventas 0 en las 20 series) y distorsiona el tipo "Religious"
train_ev = train[train["event_name_1"] != "Christmas"]
por_tipo = razon_eventos(train_ev, "event_type_1")
for t in por_tipo.index:
    lo, hi = intervalo_azar(train_ev, int(por_tipo.loc[t, "dias_evento"]))
    por_tipo.loc[t, "azar_2.5%"], por_tipo.loc[t, "azar_97.5%"] = lo, hi
por_tipo["fuera_del_azar"] = (por_tipo["razon"] < por_tipo["azar_2.5%"]) | (por_tipo["razon"] > por_tipo["azar_97.5%"])
por_tipo.round(3)

In [ ]:
por_nombre = razon_eventos(train, "event_name_1")
filas = []
for nom in por_nombre.index:
    lo, hi = intervalo_azar(train, int(por_nombre.loc[nom, "dias_evento"]), sorteos=500)
    filas.append((nom, lo, hi))
az = pd.DataFrame(filas, columns=["event_name_1", "azar_2.5%", "azar_97.5%"]).set_index("event_name_1")
por_nombre = por_nombre.join(az)
por_nombre["fuera_del_azar"] = (por_nombre["razon"] < por_nombre["azar_2.5%"]) | (por_nombre["razon"] > por_nombre["azar_97.5%"])
print("Eventos distintos:", len(por_nombre), "| fuera del azar:", int(por_nombre["fuera_del_azar"].sum()),
      "| esperados por azar puro (5 %):", round(0.05 * len(por_nombre), 1))
por_nombre[por_nombre["fuera_del_azar"]].sort_values("razon").round(3)

In [ ]:
# Razón por producto (suma de tiendas) y por tipo de evento
por_prod_tipo = razon_eventos(train_ev, ["item_id", "event_type_1"])["razon"].unstack("event_type_1")
por_prod_todos = razon_eventos(train_ev, "item_id")["razon"].rename("Todos")
tabla = por_prod_tipo.join(por_prod_todos)

fig, ax = plt.subplots(figsize=(8, 5.5))
sns.heatmap(tabla, annot=True, fmt=".2f", cmap="RdBu_r", center=1.0, vmin=0.5, vmax=1.5, ax=ax)
ax.set_title("Razón ventas reales / esperadas en días de evento, por producto")
plt.show()
print("Días de evento por tipo:", por_tipo["dias_evento"].to_dict())
celdas = por_prod_tipo.stack().sort_values()
print("Razones más bajas:", celdas.head(3).round(2).to_dict())
print("Razones más altas:", celdas.tail(3).round(2).to_dict())
print("Rango de la columna Todos:", por_prod_todos.min().round(2), "a", por_prod_todos.max().round(2))

**Observación:**
- **Navidad** (25 de diciembre): ventas 0 en las 20 series en 2013 y 2014 (tiendas cerradas). Es el único efecto contundente. Se sacó de las tablas por tipo y por producto para que no distorsione.
- Por tipo de evento (sin Navidad): Cultural 0,99, Religious 1,01, Sporting 1,02 y National 1,10. Solo National (25 días) queda apenas fuera del rango del azar (0,92 a 1,09); con 4 tipos probados, uno al borde no es evidencia firme.
- Por nombre de evento hay 30 distintos y 4 quedan fuera del azar, cuando por azar puro esperaríamos ~1,5: Christmas (0,00), NewYear (0,65), ValentinesDay (0,73) y LaborDay (1,52). Cada uno tiene solo **3 días** en `train` (Christmas, 2), y eso da poca confianza. No verifiqué si el 2013-01-01 de NewYear está afectado por las series que empiezan tarde o con rachas de ceros.
- Por producto (sin Navidad), la razón total va de 0,89 a 1,14. Hay celdas producto × tipo entre 0,61 y 1,33, pero con 9 a 28 días de evento por tipo y muchas celdas, es ruido esperable.
- Conclusión: salvo Navidad, **no hay un efecto de eventos que se pueda sostener con estos datos**. Se puede dejar `hay_evento` como variable sin esperar mucho de ella.

### 2.5 Pendientes del EDA anterior: pico de septiembre de 2013 y baja de diciembre
Quedaron anotados en el notebook 02. Los miramos con el total diario y descomponiendo por serie.

In [ ]:
# Septiembre 2013: ¿es un día puntual, una semana, o todo el mes? ¿Qué series lo explican?
ventana = total["2013-08-01":"2013-10-31"]
ref = total.rolling(28, center=True).median()   # nivel "normal" de cada momento

fig, ax = plt.subplots(figsize=(13, 4))
ventana.plot(ax=ax, label="total diario")
ref["2013-08-01":"2013-10-31"].plot(ax=ax, label="mediana móvil 28d (centrada)", ls="--")
ax.legend(); ax.set_title("Ago-Oct 2013"); plt.show()

mensual_serie = train.assign(mes=train[COL_FECHA].dt.to_period("M")).groupby(["mes"] + CLAVE_SERIE, observed=True)[COL_VENTAS].sum()
mensual_serie.index = mensual_serie.index.set_names(["mes", "store_id", "item_id"])
sep13 = mensual_serie.xs(pd.Period("2013-09"), level="mes")
sep14 = mensual_serie.xs(pd.Period("2014-09"), level="mes")
ago13 = mensual_serie.xs(pd.Period("2013-08"), level="mes")
comp = pd.DataFrame({"ago_2013": ago13, "sep_2013": sep13, "sep_2014": sep14})
comp["sep13_menos_sep14"] = comp["sep_2013"] - comp["sep_2014"]
comp["aporte_%_del_exceso"] = (100 * comp["sep13_menos_sep14"] / comp["sep13_menos_sep14"].sum()).round(1)
print("Total sep-2013:", comp['sep_2013'].sum(), "| sep-2014:", comp['sep_2014'].sum(), "| ago-2013:", comp['ago_2013'].sum())
comp.sort_values("sep13_menos_sep14", ascending=False).head(6)

In [ ]:
# ¿Qué pasó con el precio de FOODS_3_090 en CA_3 (la serie que más aporta)?
s090 = sem[(sem["store_id"] == "CA_3") & (sem["item_id"] == "FOODS_3_090")].copy()
s090["desde"] = s090["wm_yr_wk"].map(train.groupby("wm_yr_wk")[COL_FECHA].min().dt.date)
s090[(s090["desde"].astype(str) >= "2013-07-27") & (s090["desde"].astype(str) <= "2013-10-12")][["desde", "precio", "ventas", "ventas_rel"]].round(2)

In [ ]:
# Días de septiembre 2013 más altos y su evento / día de semana
sep = train[(train[COL_FECHA] >= "2013-09-01") & (train[COL_FECHA] <= "2013-09-30")]
dias = sep.groupby(COL_FECHA).agg(total=(COL_VENTAS, "sum"), evento=("event_name_1", "first"), dia=("weekday", "first"))
print("Mediana diaria sep-2013:", dias["total"].median(), "| media:", round(dias["total"].mean(), 1))
dias.sort_values("total", ascending=False).head(6)

In [ ]:
# Diciembre: total diario por año y días de caída. Se compara con noviembre y enero del mismo año/ciclo
d = total.to_frame("total")
d["anio"], d["mes"] = d.index.year, d.index.month
nov_dic = d[d["mes"].isin([11, 12])].groupby(["anio", "mes"])["total"].agg(media="mean", mediana="median", minimo="min", dias="size").round(1)
print(nov_dic.to_string())
print()
# los días más bajos de diciembre
dic = train[train[COL_FECHA].dt.month == 12].groupby(COL_FECHA).agg(total=(COL_VENTAS, "sum"), evento=("event_name_1", "first"))
dic.sort_values("total").head(8)

In [ ]:
# Diciembre sin el 25 vs octubre, y qué series explican la diferencia en 2014
sin_navidad = d[~((d["mes"] == 12) & (d.index.day == 25))]
medias = sin_navidad[sin_navidad["mes"].isin([10, 11, 12])].groupby(["anio", "mes"])["total"].mean().unstack("mes").round(1)
medias["dic/oct"] = (medias[12] / medias[10]).round(2)
print(medias.dropna().to_string())

def media_mes_serie(df, anio, mes):
    x = df[(df[COL_FECHA].dt.year == anio) & (df[COL_FECHA].dt.month == mes) & ~((mes == 12) & (df[COL_FECHA].dt.day == 25))]
    return x.groupby(CLAVE_SERIE, observed=True)[COL_VENTAS].mean()

for anio in (2013, 2014):
    caida = pd.DataFrame({"octubre": media_mes_serie(train, anio, 10), "diciembre": media_mes_serie(train, anio, 12)})
    caida["dif"] = caida["diciembre"] - caida["octubre"]
    top2 = caida["dif"].nsmallest(2)
    print(f"\n{anio}: caída oct->dic = {caida['dif'].sum():.1f} unidades/día | series que bajan: {int((caida['dif'] < 0).sum())} de {len(caida)}"
          f" | las 2 que más caen explican {100 * top2.sum() / caida['dif'].sum():.0f} %")
    print(caida.loc[top2.index].round(1).to_string())

In [ ]:
# Noviembre 2013 vs noviembre 2014: ¿por qué 2013 es tan bajo (464 vs 643 por día)?
nov = train[(train[COL_FECHA].dt.month == 11) & (train[COL_FECHA].dt.year < 2015)]
nov = nov.groupby(CLAVE_SERIE + [nov[COL_FECHA].dt.year.rename("anio")], observed=True)[COL_VENTAS].sum().unstack("anio")
nov["dif"] = nov[2014] - nov[2013]
print("Diferencia total nov-2014 menos nov-2013:", int(nov["dif"].sum()), "unidades")
nov.sort_values("dif", ascending=False).head(6)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8), sharey=True)
for ax, a in zip(axes, [2013, 2014]):
    total[f"{a}-11-01":f"{a}-12-31"].plot(ax=ax)
    ax.set_title(f"Total diario nov-dic {a}")
plt.tight_layout(); plt.show()

**Observación:**
- **Pico de septiembre 2013 (26,4 mil):** coincide con una baja de precio de `FOODS_3_090`, y las ventas son reales (no es un error de carga). En `CA_3` el precio bajó de 1,38 a 1,00 desde la semana del 2013-08-17 y las ventas semanales pasaron de ~1.100 a 2.090-3.539 (hasta 3,98 veces su promedio). Volvieron a 799 cuando el precio regresó a 1,38 (semana del 2013-09-28). `CA_3` `FOODS_3_090` explica el 82,6 % de la diferencia entre septiembre 2013 y 2014 (6.227 de 7.535 unidades) y `CA_4` `FOODS_3_090` otro 17,1 %. Ninguno de los días más altos tiene evento. Que sea una promoción es la lectura más simple de los números; no tengo el dato de campaña.
- **Baja de diciembre:** tiene dos partes distintas.
  1. El 25 de diciembre vende 0 en las dos tiendas, los dos años.
  2. Sin ese día, diciembre queda en 488 y 441 por día, 0,77 y 0,71 del nivel de octubre. Esa caída **no es general**: en 2014, dos series explican el 91 % (`CA_3` `FOODS_3_090` de 133,8 a 1,8 por día y `CA_4` `FOODS_3_090` de 27,2 a 0,0), que son las semanas a precio 1,00 sin ventas. En 2013, `CA_3` `FOODS_3_120` (de 52,3 a 4,5) y `CA_3` `FOODS_3_090` explican el 59 %.
- **Noviembre 2013 bajo (464 vs 643 en 2014):** las 6 series con mayor diferencia suman 5.523 unidades (el total es 5.364; otras series compensan hacia abajo) y son series hundidas o en racha de ceros en 2013: `CA_3` `FOODS_3_090`, `FOODS_3_587` en ambas tiendas y `FOODS_3_120` en ambas, más `CA_4` `FOODS_3_090`.
- Entonces, diciembre bajo y septiembre alto vienen de **unas pocas series con disponibilidad o promociones**, y no de una estación general.

## 3. ¿`train` y `valid` se parecen?
Es la **única** vez que se abre `valid`, y es solo para comprobar. **No se toma ninguna decisión de limpieza ni de variables con esto.**

La media diaria por serie es 29,0 en `train` y 27,0 en `valid`. ¿Es un cambio de comportamiento o es época del año? `valid` cubre del 11 de noviembre al 8 de diciembre, y vimos que noviembre y diciembre son meses bajos. Para separar las dos cosas, comparamos `valid` contra la **misma ventana calendario** de 2013 y 2014 dentro de `train`.

In [ ]:
valid = a_categoricas(cargar_tablon(RUTA_VALID), CATEGORIAS, FORMATOS)


def resumen_distribucion(df, col_ventas=COL_VENTAS):
    v = df[col_ventas]
    return pd.Series({"filas": len(v), "media": v.mean(), "mediana": v.median(), "desvio": v.std(),
                      "p90": v.quantile(0.9), "p99": v.quantile(0.99), "maximo": v.max(), "pct_ceros": 100 * (v == 0).mean(),
                      "pct_evento": 100 * df["hay_evento"].mean()})


def ventana_calendario(df, anio, desde_mmdd="11-11", hasta_mmdd="12-08", col_fecha=COL_FECHA):
    return df[(df[col_fecha] >= f"{anio}-{desde_mmdd}") & (df[col_fecha] <= f"{anio}-{hasta_mmdd}")]


comparacion = pd.DataFrame({
    "train (todo)": resumen_distribucion(train),
    "train 11-nov a 8-dic de 2013": resumen_distribucion(ventana_calendario(train, 2013)),
    "train 11-nov a 8-dic de 2014": resumen_distribucion(ventana_calendario(train, 2014)),
    "valid": resumen_distribucion(valid),
})
comparacion.round(2)

In [ ]:
# Composición de días de la semana (valid tiene exactamente 4 de cada uno; train no)
comp_dias = pd.DataFrame({
    "train": train["weekday"].value_counts(normalize=True),
    "valid": valid["weekday"].value_counts(normalize=True),
}).loc[ORDEN_DIAS].mul(100).round(1)
print(comp_dias.T.to_string())

# Media diaria por serie: train (todo), misma ventana 2013-14 y valid
def media_por_serie(df):
    return df.groupby(CLAVE_SERIE, observed=True)[COL_VENTAS].mean()

vent_prev = pd.concat([ventana_calendario(train, 2013), ventana_calendario(train, 2014)])
por_serie = pd.DataFrame({"train": media_por_serie(train), "ventana_2013_14": media_por_serie(vent_prev), "valid": media_por_serie(valid)})
por_serie["valid/train"] = por_serie["valid"] / por_serie["train"]
por_serie["valid/ventana"] = por_serie["valid"] / por_serie["ventana_2013_14"]
print()
por_tienda = por_serie.groupby("store_id")[["train", "ventana_2013_14", "valid"]].sum()
por_tienda["valid/ventana"] = por_tienda["valid"] / por_tienda["ventana_2013_14"]
print("Suma de medias por tienda:")
print(por_tienda.round(2).to_string())
print()
print("Series con valid < train:", int((por_serie['valid/train'] < 1).sum()), "de", len(por_serie),
      "| con valid < ventana 2013-14:", int((por_serie['valid/ventana'] < 1).sum()))
por_serie.round(2).sort_values("valid/ventana")

In [ ]:
from scipy import stats

# Índice semanal de valid vs train (¿cambió la forma de la semana?)
iv = indice_semanal(valid).median()
it = indice_semanal(train).median()
print(pd.DataFrame({"train": it, "valid": iv}).round(2).T.to_string())

# Dos pruebas rápidas sobre ventas diarias por fila (solo descriptivas: las filas de una misma serie no son independientes)
u = stats.mannwhitneyu(train[COL_VENTAS], valid[COL_VENTAS])
ks = stats.ks_2samp(train[COL_VENTAS], valid[COL_VENTAS])
u2 = stats.mannwhitneyu(vent_prev[COL_VENTAS], valid[COL_VENTAS])
print()
print(f"train vs valid     -> Mann-Whitney p={u.pvalue:.4f} | KS={ks.statistic:.3f}, p={ks.pvalue:.4f}")
print(f"ventana 13-14 vs valid -> Mann-Whitney p={u2.pvalue:.4f}")

In [ ]:
# Productos nuevos o precios fuera de rango en valid vs lo visto en train (solo diagnóstico)
series_train = set(map(tuple, train[CLAVE_SERIE].drop_duplicates().to_numpy()))
series_valid = set(map(tuple, valid[CLAVE_SERIE].drop_duplicates().to_numpy()))
print("Series en valid que no estaban en train:", series_valid - series_train, "| series de train ausentes en valid:", series_train - series_valid)

rango_precio = train.groupby(CLAVE_SERIE, observed=True)[COL_PRECIO].agg(["min", "max"])
v = valid.join(rango_precio, on=CLAVE_SERIE)
print("Filas de valid con precio fuera del rango de train:", int(((v[COL_PRECIO] < v["min"]) | (v[COL_PRECIO] > v["max"])).sum()))

**Observación:**
- **Nivel:** la media diaria es 29,0 en `train` y 27,0 en `valid`. La misma ventana (11-nov a 8-dic) vale 23,4 en 2013 y 28,7 en 2014 dentro de `train`. Entonces 27,0 cae dentro del rango que ya se vio en esa época; la media sola no muestra un cambio de comportamiento. Las medianas son 18 (`train`), 13 y 20,5 (ventanas 2013 y 2014) y 16 (`valid`).
- **Prueba rápida:** Mann-Whitney da p = 0,88 (train vs valid) y 0,40 (ventana 2013-14 vs valid): el valor típico no cambia. Kolmogorov-Smirnov da p = 0,0014: la forma de la distribución sí difiere un poco. Las filas de una misma serie no son independientes, así que los p-valores son orientativos.
- **Qué cambió:** `valid` tiene menos ceros (5,9 % contra 14,0 % en `train`; la ventana 2014 tiene 6,3 %). Coincide con que las rachas largas de ceros ya terminaron antes de `valid`.
- **`CA_4` baja:** suma de medias por tienda contra la ventana 2013-14: `CA_3` 1,08 veces, `CA_4` 0,86. Cinco series de `CA_4` (`FOODS_3_120`, `329`, `586`, `555`, `252`) están entre 0,36 y 0,58 de su nivel en esa ventana. `CA_4` `FOODS_3_090` va en sentido contrario (1,79).
- La composición por días de la semana es idéntica (14,3 % cada día). El índice semanal de `valid` difiere en jueves (1,09 vs 0,90) y lunes (0,83 vs 0,96), pero son solo 4 semanas y no lo leo como cambio.
- No hay series nuevas ni precios fuera del rango de `train`.
- **Conclusión:** `valid` se parece a `train` en nivel y estructura, y la diferencia de la media es compatible con la época del año. Hay dos señales para seguir de cerca: `CA_4` más flojo y menos ceros. No se tomó ninguna decisión con esta comparación.

## Resumen y qué sigue

- **Cobertura:** completa y sin huecos; las dos series de `FOODS_3_288` empiezan tarde (ya limpiado).
- **Ceros:** vienen en rachas largas (hasta 361 días) con el precio cargado. No son demanda "normal". Navidad vende 0 en toda la cadena.
- **Extremos:** 37 valores (0,18 %), casi todos de `FOODS_3_090` en agosto-septiembre 2013. No se tocan.
- **Precio:** sin relación consistente con ventas. El precio 1,00 en `FOODS_3_090` de `CA_3` coincide con ventas casi nulas desde fines de 2013 (hipótesis: producto no disponible; no verificada).
- **Patrones:** semana muy marcada (sábado y domingo +17 % y +23 % sobre el promedio); año débil (julio alto, diciembre bajo); eventos sin efecto sostenible salvo Navidad.
- **Pendientes del notebook anterior:** el pico de septiembre coincide con una baja de precio de `FOODS_3_090`; la baja de diciembre es Navidad más dos o tres series hundidas.
- **`valid`:** se parece a `train`; la diferencia de media es compatible con la época del año.

**Qué falta decidir (no lo decidí yo; queda en `decisions.md` como pendiente)**
1. Qué hacer con las rachas largas de ceros y las semanas hundidas: dejarlas, marcarlas con una variable o excluirlas del entrenamiento.
2. Si el precio 1,00 es un relleno o un descuento: confirmarlo con quien conozca el origen de los datos.
3. Cómo tratar el 25 de diciembre (tiendas cerradas): variable propia o filtrar.
4. Cómo representar la promoción de agosto-septiembre 2013 (precio relativo a la serie).

**Siguiente paso:** feature engineering sobre `train`: rezagos 7, 14 y 28, medias móviles, precio relativo a la mediana de la serie y marca de racha de ceros.